# dNaty v5 — CIFAR-10 no Colab (GPU T4)
> **FastDataset**: 10K em RAM, zero I/O por geração. ~20 min total.
> Todos os outros experimentos já rodaram localmente (MNIST 98.70%, FashionMNIST 90.00%, CL BWT=-0.20).
> **Aqui só roda o CIFAR-10** — o único que precisa de GPU.

**Ordem:**
1. Célula 1 — verificar GPU
2. Célula 2 — instalar dependências
3. Célula 3 — upload do `dnaty_code.zip`
4. Célula 8b — CIFAR-10 (~20 min)
5. Célula 10 — download do resultado

## Célula 1 — Verificar GPU

In [1]:
import torch
print('GPU disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else:
    print('AVISO: sem GPU — mude Runtime > Change runtime type > T4 GPU')

GPU disponível: False
AVISO: sem GPU — mude Runtime > Change runtime type > T4 GPU


## Célula 2 — Instalar dependências

In [2]:
!pip install torch torchvision numpy scipy matplotlib tqdm -q
print('Dependências instaladas')

Dependências instaladas


## Célula 3 — Upload do código dNaty
> Faça upload do arquivo `dnaty_code.zip`

In [3]:
from google.colab import files
import zipfile, os, sys

print('Selecione o arquivo dnaty_code.zip...')
uploaded = files.upload()

for fname in uploaded:
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('.')
    print(f'Descompactado: {fname}')

os.makedirs('results', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Garantir que o path está correto
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Verificar arquivos essenciais
files_needed = [
    'dnaty/__init__.py',
    'dnaty/operators/mutations.py',
    'dnaty/evolution/evolver.py',
    'dnaty/experiments/exp1_mnist.py',
    'dnaty/experiments/exp2_cifar.py',
    'dnaty/experiments/exp3_cl.py',
]
missing = [f for f in files_needed if not os.path.exists(f)]
if missing:
    print('ERRO — arquivos faltando:', missing)
else:
    # Teste rápido de import
    from dnaty.operators.mutations import OPERATORS, OPERATOR_FNS
    print(f'OK! {len(OPERATORS)} operadores carregados:', OPERATORS)


ModuleNotFoundError: No module named 'google.colab'

## Célula 7 — Configuração dos Experimentos

In [ ]:
# CONFIG: completa = G=50, N=20, T_local=5, 10K treino / 10K val completo
# CONFIG: rapida   = G=15, N=6,  T_local=2, 3K treino / 1K val
#
# NOTA: usar 60K de treino com N=20 e T_local=5 leva ~8h no T4.
# 10K de treino é o padrão em NAS — representativo e ~45 min.

CONFIG = 'completa'

if CONFIG == 'completa':
    N_GENERATIONS = 30
    N_POP = 8
    T_LOCAL = 2
    TRAIN_SUBSET = 3000  # 3K — cabe em 3h de GPU T4
    VAL_SUBSET = None     # val completo 10K — métricas honestas
    print('Config COMPLETA: G=30, N=8, T_local=2, 3K treino / 10K val')
    print('Tempo estimado: ~2.5h com GPU T4 (3 experimentos)')
else:
    N_GENERATIONS = 15
    N_POP = 6
    T_LOCAL = 2
    TRAIN_SUBSET = 3000
    VAL_SUBSET = 1000
    print('Config RAPIDA: G=15, N=6, T_local=2, 3K treino / 1K val')
    print('Tempo estimado: ~5 min com GPU T4')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


## Célula 8 — Experimento 1: MNIST + FashionMNIST

In [ ]:
import sys, importlib
if '.' not in sys.path:
    sys.path.insert(0, '.')

import dnaty.experiments.exp1_mnist as exp1_mod
importlib.reload(exp1_mod)

exp1_mod.N_GENERATIONS = N_GENERATIONS
exp1_mod.N_POP = N_POP
exp1_mod.T_LOCAL = T_LOCAL
exp1_mod.TRAIN_SUBSET = TRAIN_SUBSET
exp1_mod.VAL_SUBSET = VAL_SUBSET

print('Iniciando Experimento 1 (MNIST + FashionMNIST)...')
results_exp1 = exp1_mod.main()
print('Experimento 1 concluído!')


## Célula 8b — Experimento 2: CIFAR-10 (GPU — ~20 min no T4)
> **FastDataset v5**: 10K em RAM, zero I/O por geração. ~30s/geração no T4.

In [ ]:
import sys, time, json, importlib
sys.path.insert(0, '.')
import dnaty.experiments.exp2_cifar as exp2_mod
importlib.reload(exp2_mod)

# Config v5 — FastDataset, BatchNorm, cosine annealing
exp2_mod.N_GENERATIONS = 20
exp2_mod.N_POP = 8
exp2_mod.T_LOCAL = 3
exp2_mod.BATCH_SIZE = 256
exp2_mod.CIFAR_TRAIN_SUBSET = 10000  # 10K em RAM — rápido
exp2_mod.SEEDS = [0, 1, 2]

print('Iniciando Experimento 2 (CIFAR-10 v5 — FastDataset)...')
print('Estimativa: ~20 min no T4')
t0 = time.time()
results_exp2 = exp2_mod.main()
print(f'Experimento 2 concluído em {(time.time()-t0)/60:.1f} min!')


## Célula 9 — Experimento 3: Split-MNIST Continual Learning (FastDataset)

In [ ]:
import dnaty.experiments.exp3_cl as exp3_mod
importlib.reload(exp3_mod)

exp3_mod.N_EPOCHS_CL = 15
exp3_mod.TRAIN_SUBSET_CL = None  # dataset completo por task
exp3_mod.SEEDS = [0, 1, 2]

print('Iniciando Experimento 3 (CL v5 — FastDataset)...')
t0 = time.time()
results_exp3 = exp3_mod.main()
print(f'Experimento 3 concluído em {(time.time()-t0)/60:.1f} min!')


## Célula 10 — Gerar Relatório e Download

In [ ]:
import os
from dnaty.analysis.report import generate_report

# Gerar DNATY_RESULTS.md
generate_report(results_dir='results', output_path='DNATY_RESULTS.md')

# Download dos arquivos
from google.colab import files as colab_files

to_download = [
    'results/exp1_results.json',
    'results/exp2_cifar10_results.json',
    'results/exp3_cl_results.json',
    'DNATY_RESULTS.md',
]

for f in to_download:
    if os.path.exists(f):
        colab_files.download(f)
        print(f'Download: {f}')
    else:
        print(f'AVISO: {f} não encontrado (experimento pode não ter rodado)')

print('Pronto!')


## Célula 11 — Gráfico de Convergência (opcional)

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np

# Carregar resultados
e1_path = 'results/exp1_results.json'
if not os.path.exists(e1_path):
    print('Rode o Experimento 1 primeiro (Célula 8)')
else:
    with open(e1_path) as f:
        e1_data = json.load(f)

    plt.style.use('dark_background')
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.patch.set_facecolor('#05070f')

    colors = {'MNIST': '#4f8fff', 'FashionMNIST': '#50e0a0'}

    for idx, ds in enumerate(['MNIST', 'FashionMNIST']):
        ax_acc = axes[0][idx]
        ax_delta = axes[1][idx]
        ax_acc.set_facecolor('#080c18')
        ax_delta.set_facecolor('#080c18')

        hist = e1_data[ds]['dnaty'][0]['history']
        gens = [h['gen'] for h in hist]
        acc  = [h['best_acc'] * 100 for h in hist]
        dg   = [h['delta_grad'] for h in hist]
        dm   = [h['delta_mem'] for h in hist]

        ax_acc.plot(gens, acc, color=colors[ds], linewidth=2.5, marker='o', markersize=4)
        ax_acc.axhline(y=97.3 if ds == 'MNIST' else 89.1, color='#f0c060',
                       linestyle='--', alpha=0.5, label='Meta paper')
        ax_acc.set_title(f'{ds} — Acurácia (seed=0)', color='#d8e4f8', fontsize=11)
        ax_acc.set_ylabel('Acurácia (%)', color='#8898bb')
        ax_acc.tick_params(colors='#4a5a7a')
        ax_acc.legend(fontsize=8)
        ax_acc.grid(alpha=0.1)

        ax_delta.plot(gens, dg, color='#50e0a0', linewidth=2, marker='o', markersize=3, label='δ_grad')
        ax_delta.plot(gens, dm, color='#f0c060', linewidth=2, marker='o', markersize=3, label='δ_mem')
        ax_delta.axhline(y=0, color='#ff6060', linestyle='--', alpha=0.4, label='≥ 0 (Teorema 1)')
        ax_delta.set_title(f'{ds} — δ_grad e δ_mem (Teorema 1)', color='#d8e4f8', fontsize=11)
        ax_delta.set_ylabel('δ', color='#8898bb')
        ax_delta.set_xlabel('Geração', color='#8898bb')
        ax_delta.tick_params(colors='#4a5a7a')
        ax_delta.legend(fontsize=8)
        ax_delta.grid(alpha=0.1)

    plt.suptitle('dNaty — Convergência Real · Validação Teorema 1', color='#d8e4f8', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('dnaty_convergence.png', dpi=150, bbox_inches='tight', facecolor='#05070f')
    plt.show()

    from google.colab import files as colab_files
    colab_files.download('dnaty_convergence.png')
    print('Gráfico salvo!')


---
## Instruções

1. **Abrir no Colab:** `File > Upload notebook`
2. **Ativar GPU:** `Runtime > Change runtime type > T4 GPU > Save`
3. **Upload do código:** zipar a pasta `dnaty/` → `dnaty_code.zip` → upload na Célula 3
4. **Rodar em ordem:** Células 1 → 2 → 3 → 7 → 8 → 8b → 9 → 10
5. **Baixar resultados:** Célula 10 faz download automático
6. **Localmente:** copiar JSONs para `results/` e rodar `python update_data.py`

**Resultados esperados (config completa, GPU T4):**
- MNIST: ~97.3% ± 0.4%
- FashionMNIST: ~89.1% ± 0.6%
- CIFAR-10: ~75% (50K amostras, G=50)
- Split-MNIST BWT: ~−0.031 (vs EWC −0.089)
